In [5]:
#了解ASII,unicode和utf-8编码
print('a')  #对应的ASCII码是97
print(ord('a'))  #ord函数可以查看字符对应的ASCII码
print(chr(97))  #chr函数可以查看ASCII码对应的字符
print('中')  #对应的Unicode编码是20013
print(ord('中'))  #ord函数可以查看字符对应的Unicode编码
print(chr(20013))  #chr函数可以查看Unicode编码对应的字符

#unicode只是解决了身份位置，但是没有解决怎么在内存里存储的问题
# 接下来就需要用到utf-8编码，前缀识别法
print('中'.encode('utf-8'))  #utf-8编码会把一个字符转换成多个字节存储
print(b'\xe4\xb8\xad'.decode('utf-8'))  #可以通过decode方法把字节转换成字符，这里是三个字节

a
97
a
中
20013
中
b'\xe4\xb8\xad'
中


In [1]:
# 以鲁为例子
print('鲁')
print(ord('鲁'))  #查看Unicode编码
print(bin(ord('鲁')))  #查看Unicode编码的二进制表示 0b1001110010000001这里为16bit 0bxxx
print(hex(ord('鲁')))  #查看Unicode编码的十六进制表示 0x9c81
print('鲁'.encode('utf-8'))  #查看utf-8编码 b'\xe9\xb2\x81'

鲁
40065
0b1001110010000001
0x9c81
b'\xe9\xb2\x81'


In [19]:
import regex

# 使用 [ ] 将空格包裹起来，防止被 VERBOSE 模式忽略
GPT2_PAT = regex.compile(r"""
    's|'t|'re|'ve|'m|'ll|'d       # 英语缩写
    | [ ]?\p{L}+                  # 字母序列（前面可能有空格 -> 用 [ ]? 表示）
    | [ ]?\p{N}+                  # 数字序列（前面可能有空格）
    | [ ]?[^\s\p{L}\p{N}]+        # 其他非字母数字字符序列（标点等）
    | \s+(?!\S)                   # 后面没有非空白字符的空白（通常是行尾空白）
    | \s+                         # 其他空白字符
""", regex.VERBOSE)


def gpt_pre_tokenize(text):
    tokens = GPT2_PAT.findall(text)
    return tokens


# 测试
text = "banana"
tokens = gpt_pre_tokenize(text)
for word in tokens:
   word_ids = list(word.encode("utf-8"))#把每一个单词的字节编码组合为列表
   print(word_ids)
print(tokens)

[98, 97, 110, 97, 110, 97]
['banana']


In [16]:
import collections
import regex

class BPE_From_Scratch:
    """
    步骤1，字节化和初始化，初始词表为0-255，每一个字节都是一个独立的token
    步骤2，构建统计字典，统计预分词后的token对出现的频率
    步骤3，寻找最高频率的字节对，合并为一个新的token，更新词表和文本
    步骤4，重复步骤2和3，直到达到预设的词表大小或者没有更多的token对可以合并
    性能优化：采用倒排索引来加速频率统计和更新文本
    终止条件：达到预设的词表大小或者没有更多的token对可以合并
    结果输出：最终的词表和编码后的文本
    """

    def __init__(self, vocab_size=1000):
        """
        初始化BPE模型
        这里的vocab_size是指最终想要的词表大小
        由于初始词表已经有256个token（0-255的字节），
        因此实际需要通过合并生成的新token数量是vocab_size - 256
        这样可以确保最终的词表大小达到预期
        当然，如果vocab_size小于256，则表示不需要进行任何合并操作
        直接使用初始的字节词表即可
        :param self:
        :param vocab_size: 词表大小
        :return: 
        """
        self.vocab_size = vocab_size
        # 初始词表：0-255 的 ASCII值
        # 格式: {token_id: bytes}
        self.vocab = {i: bytes([i]) for i in range(256)}
        # 记录合并规则：(token_a, token_b) -> new_token_id
        self.merges = {}
        self.next_token_id = 256
        self.GPT2_PAT = regex.compile(r"""
                        's|'t|'re|'ve|'m|'ll|'d       # 英语缩写
                        | [ ]?\p{L}+                  # 字母序列（前面可能有空格 -> 用 [ ]? 表示）
                        | [ ]?\p{N}+                  # 数字序列（前面可能有空格）
                        | [ ]?[^\s\p{L}\p{N}]+        # 其他非字母数字字符序列（标点等）
                        | \s+(?!\S)                   # 后面没有非空白字符的空白（通常是行尾空白）
                        | \s+                         # 其他空白字符
                    """, regex.VERBOSE)
    def gpt_pre_tokenize(self,text):
        tokens = GPT2_PAT.findall(text)
        return tokens

    def build_dict(selfs, words):
        """
        构建统计字典，统计预分词后的token对出现的频率
        :param selfs:               
        :param words: 预分词的列表
        :return: 返回频率字典，格式为单词的字节表示:频率
        """
        word_frep = collections.defaultdict(int)  #创建一个空字典,记录每个单词出现的频率，该字典和普通的字典类似，但是当访问一个不存在的键时，会自动创建该键并赋值为默认值（这里是0）   
        for word in words:
            0 = tuple(word.encode("utf-8"))  #字典的键是单词的字节表示,元组，值是该单词出现的频率
            word_frep[word_bytes] += 1
        return word_frep  #返回频率字典，格式为单词的字节表示:频率
    
    def update_word_freqs(self, pair, new_token_id, word_freqs):
        """
        helper function: 将字典中所有出现的 pair 替换为 new_token_id
        """
        new_word_freqs = collections.defaultdict(int)
        
        # 遍历旧字典
        for word_tuple, freq in word_freqs.items():
            # 优化：如果当前单词不包含 pair 的第一个字符，肯定不需要替换，直接保留
            if pair[0] not in word_tuple:
                new_word_freqs[word_tuple] += freq
                continue
            
            # 开始查找并替换
            new_tuple = []
            i = 0
            while i < len(word_tuple):
                # 检查是否匹配我们要合并的 pair
                if i < len(word_tuple) - 1 and word_tuple[i] == pair[0] and word_tuple[i+1] == pair[1]:
                    new_tuple.append(new_token_id) # 写入新 ID
                    i += 2 # 指针跳过两个旧字符
                else:
                    new_tuple.append(word_tuple[i]) # 写入旧字符
                    i += 1 # 指针正常前进
            
            # 将处理后的新元组存入新字典
            new_word_freqs[tuple(new_tuple)] += freq
            
        return new_word_freqs
    
    def train(self, text):
        """
        步骤3，寻找最高频率的字节对，合并为一个新的token，更新词表和文本
        :param text: 训练集文本
        :return: self.vocab, self.merges
        """
        # 预分词 GPT2_PAT 正则使用
        words = gpt_pre_tokenize(text)
        # 构建频率字典
        word_frep = self.build_dict(words)
        # for _word, freq in word_frep.items():
        #     print(_word, freq)  #输出每个单词的字节表示及其频率
        while len(self.vocab) < self.vocab_size:#只要词表没满，就一直训练
            #统计字节对出现的频率，每一个预分词后的词的相邻字节对，不会发生跨词的情况
            pairs = collections.defaultdict(int)
            for word_tuple, freq in word_frep.items():#遍历频率字典 格式如(97, 97) 1，第一个是元组，第二个是频率
                for i in range(len(word_tuple)-1):#遍历单词的每一个字节
                    pair = (word_tuple[i],word_tuple[i+1])# 取出相邻的字节对
                    pairs[pair] += freq  #加权统计（加上单词出现的频率）,因为字典是唯一的，但是字典的value是出现的频率，所以要加上频率
            #寻找最高频率的字节对
            if not pairs:
                # print("[Stop] 没有更多的 token 对可以合并")
                break
            best_pair = max(pairs, key=pairs.get)#寻找最大值对应的键
            best_count = pairs[best_pair]
            print(f"Merge: {best_pair} -> Count: {best_count}")
            #更新词表
            new_token_id = self.next_token_id 
            self.vocab[new_token_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.merges[best_pair] = new_token_id
            self.next_token_id += 1#更新id
            #打印更新后的词表
            # print(f"Updated Vocab Size: {len(self.vocab)}")
            # 更新文本数据 (最麻烦但必须做的一步)
            # -------------------------------------------------------
            # 你需要把 word_frep 里的 (108, 108) 全部替换成 (256)
            # 否则下一轮循环进来，pairs 统计到的还是 108, 108
            word_frep = self.update_word_freqs(best_pair, new_token_id, word_frep)
            # print(f"Updated Vocab Size: {len(self.vocab)}")
        return self.vocab, self.merges
    def encode(self, text):
        # 1. 还是先预分词，不然会把单词连起来
        words = self.gpt_pre_tokenize(text)
        ids = []

        for word in words:
            # 把每个单词转成基础 ID 列表
            word_ids = list(word.encode("utf-8"))
            print(word_ids)
            while len(word_ids) >= 2:
                # --- 核心：寻找当前单词里所有能合并的 pair ---
                # 比如 word_ids = [h, e, l, l, o]
                # 这一步会找出所有在 self.merges 里出现过的相邻对
                stats = {}
                for i in range(len(word_ids) - 1):
                    pair = (word_ids[i], word_ids[i+1])
                    if pair in self.merges:
                        # 记录这个 pair 对应的 新ID (作为优先级的依据)
                        stats[pair] = self.merges[pair]

                # 如果这就没找到任何能合并的，说明这个词处理完了
                if not stats:
                    break

                # --- 决策：谁的 ID 最小（最早被 merge 的），就先合并谁 ---
                # min 函数会根据 value (新ID) 来找最小的 key (pair)
                pair_to_merge = min(stats, key=stats.get)
                new_id = self.merges[pair_to_merge]

                # --- 执行合并 (跟 train 里的逻辑一样) ---
                new_ids = []
                i = 0
                while i < len(word_ids):
                    if i < len(word_ids) - 1 and word_ids[i] == pair_to_merge[0] and word_ids[i+1] == pair_to_merge[1]:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(word_ids[i])
                        i += 1
                word_ids = new_ids # 更新当前单词的 ID 列表
            
            # 把处理完的单词加入最终结果
            ids.extend(word_ids)
            
        return ids

    def decode(self, ids):
        # 1. 创建一个反向词表: id -> bytes
        # 注意：这里需要把 vocab 里的 bytes 拼起来
        # 但是为了简单，我们可以直接用现有的 vocab，因为 vocab 已经是 {id: bytes} 格式了
        # 只需要把列表里的 bytes 拼接起来即可

        # 修正：你的 vocab 里的 value 是 bytes，比如 b'a' 或者 b'ab'
        tokens = b"".join([self.vocab[idx] for idx in ids])

        # 转回字符串
        return tokens.decode("utf-8", errors="replace")           
        

开始训练 BPE，目标词表大小: 300
训练完成，最终词表大小: 279
Encoded: [259, 264]
Decoded: Hello world
模型已保存到 my_tokenizer.json
模型已从 my_tokenizer.json 加载
Reloaded Decode: Hello world


In [15]:
# 你的测试代码
bpe = BPE_From_Scratch()
ids = bpe.encode(" ab aa") 
# 注意原文里有空格！
# 预期过程：
# " ab" -> (32, 97, 98) -> (256, 98) -> (257) 
# " aa" -> (32, 97, 97) -> (256, 97) -> (258)
# 最终 ids 应该很短，比如 [257, 258]

print(f"编码ID: {ids}")
print(f"解码回原文: {bpe.decode(ids)}")

[32, 97, 98]
[32, 97, 97]
编码ID: [32, 97, 98, 32, 97, 97]
解码回原文:  ab aa


In [12]:
text_data = "aa ab aa ab ac aa ad aa ab"
bpe = BPE_From_Scratch()
bpe.train(text_data)
print(bpe.encode("abc bcac ccc"))

Merge: (32, 97) -> Count: 8
Merge: (256, 98) -> Count: 3
Merge: (256, 97) -> Count: 3
Merge: (97, 97) -> Count: 1
Merge: (256, 99) -> Count: 1
Merge: (256, 100) -> Count: 1
[97, 98, 99]
[32, 98, 99, 97, 99]
[32, 99, 99, 99]
[97, 98, 99, 32, 98, 99, 97, 99, 32, 99, 99, 99]


In [42]:
text_data = "aa ab aa ab ac aa ad aa ab"
bpe = BPE_From_Scratch()
# bpe.train(text_data)
word_frep = bpe.build_dict(gpt_pre_tokenize(text_data))
print(word_frep)  #输出每个单词的字节表示及其频
pairs = collections.defaultdict(int)
for word_tuple, freq in word_frep.items():#遍历频率字典 格式如(97, 97) 1，第一个是元组，第二个是频率
    for i in range(len(word_tuple)-1):#遍历单词的每一个字节
        pair = (word_tuple[i],word_tuple[i+1])# 取出相邻的字节对
        pairs[pair] += freq  #加权统计（加上单词出现的频率）,因为字典是唯一的，但是字典的value是出现的频率，所以要加上频率
for _pair, count in pairs.items():
    print(f"Pair: {_pair} -> Count: {count}")

defaultdict(<class 'int'>, {(97, 97): 1, (32, 97, 98): 3, (32, 97, 97): 3, (32, 97, 99): 1, (32, 97, 100): 1})
Pair: (97, 97) -> Count: 4
Pair: (32, 97) -> Count: 8
Pair: (97, 98) -> Count: 3
Pair: (97, 99) -> Count: 1
Pair: (97, 100) -> Count: 1


In [34]:
vocab = {i: bytes([i]) for i in range(256)}
print(vocab[97])  #b'a'
# text_data = "aa ab aa ab ac aa ad aa ab"
# words = gpt_pre_tokenize(text_data)
# print(words)
# bpe = BPE_From_Scratch()
# word_freqs = bpe.build_dict(words)
# print(word_freqs)  #输出每个单词的字节表示及其频率

b'a'


In [ ]:
import collections


class BPE_From_Scratch:
    """
    步骤1，字节化和初始化，初始词表为0-255，每一个字节都是一个独立的token
    步骤2，构建统计字典，统计预分词后的token对出现的频率
    步骤3，寻找最高频率的字节对，合并为一个新的token，更新词表和文本
    步骤4，重复步骤2和3，直到达到预设的词表大小或者没有更多的token对可以合并
    性能优化：采用倒排索引来加速频率统计和更新文本
    终止条件：达到预设的词表大小或者没有更多的token对可以合并
    结果输出：最终的词表和编码后的文本
    """

    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
        # 初始词表：0-255 的 ASCII/Byte 值
        # 格式: {token_id: bytes}
        self.vocab = {i: bytes([i]) for i in range(256)}
        # 记录合并规则：(token_a, token_b) -> new_token_id
        self.merges = {}
        self.next_token_id = 256

    def train(self, text):
        # -------------------------------------------------------------
        # 步骤1，字节化和初始化
        # -------------------------------------------------------------
        # 预分词 GPT2_PAT 正则使用
        words = gpt_pre_tokenize(text)

        # 将文本转换为 token 列表的频率字典
        # 例如: "hello" -> (104, 101, 108, 108, 111)
        word_freqs = collections.defaultdict(int)
        for word in words:
            # 将单词转为 UTF-8 字节流，再转为整数元组
            word_bytes = tuple(word.encode("utf-8"))
            word_freqs[word_bytes] += 1

        print(f"[Info] 初始词表大小: {len(self.vocab)}")
        print(f"[Info] 唯一单词数量: {len(word_freqs)}")

        # -------------------------------------------------------------
        # 步骤4，重复步骤2和3
        # -------------------------------------------------------------
        while len(self.vocab) < self.vocab_size:
            # -------------------------------------------------------------
            # 步骤2，构建统计字典
            # -------------------------------------------------------------
            pairs = collections.defaultdict(int)
            for word_tuple, freq in word_freqs.items():
                # 遍历当前单词中的所有相邻对
                for i in range(len(word_tuple) - 1):
                    pair = (word_tuple[i], word_tuple[i + 1])
                    pairs[pair] += freq  # 加权统计（乘以单词出现的频率）

            # -------------------------------------------------------------
            # 终止条件检查
            # -------------------------------------------------------------
            if not pairs:
                print("[Stop] 没有更多的 token 对可以合并")
                break

            # -------------------------------------------------------------
            # 步骤3，寻找最高频率的字节对，合并
            # -------------------------------------------------------------
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]

            # 更新词表
            new_token_id = self.next_token_id
            self.vocab[new_token_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.merges[best_pair] = new_token_id
            self.next_token_id += 1

            # 更新文本（这里的“文本”是 word_freqs 字典）
            # 性能优化：只更新包含 best_pair 的单词，而不是全量扫描
            new_word_freqs = collections.defaultdict(int)
            for word_tuple, freq in word_freqs.items():
                # 如果这个单词里不包含我们要合并的对，直接保留
                if best_pair[0] not in word_tuple:
                    # 简单的过滤优化，实际可以使用更复杂的倒排索引
                    new_word_freqs[word_tuple] = freq
                    continue

                # 执行合并操作
                new_tuple = []
                i = 0
                while i < len(word_tuple):
                    # 检查是否匹配 best_pair
                    if i < len(word_tuple) - 1 and word_tuple[i] == best_pair[0] and word_tuple[i + 1] == best_pair[1]:
                        new_tuple.append(new_token_id)
                        i += 2  # 跳过两个
                    else:
                        new_tuple.append(word_tuple[i])
                        i += 1
                new_word_freqs[tuple(new_tuple)] = freq

            word_freqs = new_word_freqs

            # 打印日志
            if len(self.vocab) % 5 == 0:  # 每合并5次打印一次
                print(f"Merge: {best_pair} -> {new_token_id} (Count: {best_count})")

        # -------------------------------------------------------------
        # 结果输出
        # -------------------------------------------------------------
        print(f"\n[Done] 训练完成。最终词表大小: {len(self.vocab)}")
        return self.vocab, self.merges

    def encode(self, text):
        """
        使用训练好的 merges 规则对新文本进行编码
        """
        words = text.split()
        encoded_ids = []

        for word in words:
            # 初始状态：转为字节 ID
            word_ids = list(word.encode("utf-8"))

            while len(word_ids) >= 2:
                # 寻找当前单词中，所有可能的 pair，找到在 merges 中优先级最高（最早加入）的那个
                # 注意：实际 BPE 实现中，应该按照 merges 的加入顺序来合并
                stats = {}
                for i in range(len(word_ids) - 1):
                    pair = (word_ids[i], word_ids[i + 1])
                    if pair in self.merges:
                        # 我们需要知道这个 pair 是第几个被 merge 的，这里简化逻辑
                        stats[pair] = pair  # 简单占位，实际应存 priority

                if not stats:
                    break

                # 这里为了简单演示，假设只要在 merges 里就合并（实际应选最早 merge 的）
                # 找到当前单词中能合并的一对（这里简单取第一个匹配的）
                pair_to_merge = list(stats.keys())[0]
                new_id = self.merges[pair_to_merge]

                # 执行替换
                new_ids = []
                i = 0
                while i < len(word_ids):
                    if i < len(word_ids) - 1 and word_ids[i] == pair_to_merge[0] and word_ids[i + 1] == pair_to_merge[
                        1]:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(word_ids[i])
                        i += 1
                word_ids = new_ids

            encoded_ids.extend(word_ids)

        return encoded_ids


# -------------------------------------------------------------
# 测试代码
# -------------------------------------------------------------
if __name__ == "__main__":
    # 模拟一段重复度高的数据，方便观察合并效果
    # "aa" -> 97, 97 -> 合并
    text_data = "aa ab aa ab ac aa ad aa ab"

    # 实例化 BPE
    bpe = BPE_From_Scratch(vocab_size=260)  # 初始256，我们只想多训练4个词

    # 训练
    vocab, merges = bpe.train(text_data)

    # 打印部分结果
    print("\n--- Merges (合并规则) ---")
    for pair, new_id in merges.items():
        print(f"Pair {pair} -> New ID {new_id}")

    # 测试编码
    test_str = "ab aa ac"
    encoded = bpe.encode(test_str)
    print(f"\n--- Encode Test ---")
    print(f"原文: {test_str}")
    print(f"编码 ID: {encoded}")

In [20]:
# 训练数据下载
import requests
import os

def download_shakespeare():
    # 这是 Andrej Karpathy 提供的原始 txt 文件链接
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    file_path = "input.txt"

    if not os.path.exists(file_path):
        print(f"正在下载 {file_path}...")
        try:
            r = requests.get(url)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(r.text)
            print(f"下载成功！文件大小: {len(r.text) / 1024 / 1024:.2f} MB")
        except Exception as e:
            print(f"下载失败: {e}")
            print("请检查网络，或者尝试方法 2 手动下载")
    else:
        print(f"{file_path} 已经存在，跳过下载。")

if __name__ == "__main__":
    download_shakespeare()

正在下载 input.txt...
下载成功！文件大小: 1.06 MB


In [21]:
# 手写Embedding

import torch
import torch.nn as nn

class ManualEmbedding(nn.Module):
    """
    纯手写 Embedding 层
    本质：维护一个可学习的大矩阵，并支持通过索引取出对应的行。
    """
    def __init__(self, vocab, d_model):
        # 必须调用父类初始化，否则 PyTorch 无法识别这是一个神经网络模块
        super().__init__()
        
        # -----------------------------------------------------------------
        # 步骤 1: 创建“货架” (Create Weight Matrix)
        # -----------------------------------------------------------------
        # torch.randn(...): 
        #   在内存中开辟一块空间，生成形状为 [词表大小, 向量维度] 的随机数。
        #   此时它还只是普通的“数据”。
        #
        # nn.Parameter(...): 
        #   这是点睛之笔！它给普通数据穿上了一层“马甲”。
        #   作用：告诉 PyTorch 引擎，“这个张量是模型的可训练参数”。
        #   后果：1. 它会自动出现在 model.parameters() 里。
        #         2. 反向传播时，优化器会计算它的梯度并更新它。
        self.weight = nn.Parameter(torch.randn(vocab, d_model))
        
        # -----------------------------------------------------------------
        # 步骤 2: 装修“货架” (Initialization)
        # -----------------------------------------------------------------
        # nn.init.normal_(...):
        #   我们不能直接用 randn 生成的原始随机数，因为它们方差可能太大（默认是1）。
        #   这里强制把数值重置为：均值 0，标准差 0.02。
        #   原因：这是 GPT-2/BERT 等模型的标准做法，数值小一点，模型学得稳一点。
        #   注意：函数名末尾的下划线 _ 表示 "In-place" 操作，直接修改 self.weight 本身。
        nn.init.normal_(self.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        # -----------------------------------------------------------------
        # 步骤 3: 机械臂取货 (Lookup / Advanced Indexing)
        # -----------------------------------------------------------------
        # 输入 idx: 
        #   形状通常是 [Batch_Size, Seq_Len]，里面的元素是整数 ID (如 5, 102, 0)。
        #
        # 操作 self.weight[idx]:
        #   这是 PyTorch/NumPy 的“高级索引”功能。
        #   它不是矩阵乘法，它是“物理搬运”。
        #   逻辑：如果 idx 里是 5，就去 self.weight 的第 5 行把整行数据拷贝出来。
        #
        # 输出形状:
        #   [Batch_Size, Seq_Len, embedding_dim]
        #   原来的每个 ID 变成了一个向量，维度增加了一级。
        return self.weight[idx]

In [23]:
def verify_equivalence():
    vocab_size = 1000
    n_embd = 64
    batch_size = 2
    seq_len = 5

    # --- 选手 A: 官方库 ---
    auto_emb = nn.Embedding(vocab_size, n_embd)
    
    # --- 选手 B: 手写版 ---
    manual_emb = ManualEmbedding(vocab_size, n_embd)

    # 【关键步骤】强制复制权重，确保起跑线一致
    # clone() 是为了让内存独立，互不影响
    manual_emb.weight.data = auto_emb.weight.data.clone()

    # 准备假数据
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

    # --- 第一回合：比前向传播 (输出值) ---
    out_auto = auto_emb(input_ids)
    out_manual = manual_emb(input_ids)

    # torch.allclose 用来比较两个浮点数张量是否“足够接近”
    forward_match = torch.allclose(out_auto, out_manual)
    print(f"前向传播结果一致? {forward_match}")

    # --- 第二回合：比反向传播 (梯度值) ---
    # 模拟一个 Loss，比如求和
    loss_auto = out_auto.sum()
    loss_manual = out_manual.sum()

    # 反向传播
    loss_auto.backward()
    loss_manual.backward()

    # 比较计算出来的梯度
    grad_match = torch.allclose(auto_emb.weight.grad, manual_emb.weight.grad)
    print(f"反向传播梯度一致? {grad_match}")

    # 打印前几个数值看看
    print("\n--- 抽样检查梯度值 (前3个) ---")
    print("官方库 Grad:", auto_emb.weight.grad.view(-1)[:3].tolist())
    print("手写版 Grad:", manual_emb.weight.grad.view(-1)[:3].tolist())
verify_equivalence()

前向传播结果一致? True
反向传播梯度一致? True

--- 抽样检查梯度值 (前3个) ---
官方库 Grad: [0.0, 0.0, 0.0]
手写版 Grad: [0.0, 0.0, 0.0]


In [29]:
import torch
vocab = 1000
d_model = 64
i=torch.arange(0,d_model,2)
i.unsqueeze(0).shape

torch.Size([1, 32])

In [31]:
pos = torch.arange(0,vocab).unsqueeze(1)
pos.shape

torch.Size([1000, 1])

In [ ]:
import torch
import numpy as np
# 假设你的类都保存在 ManualEmbedding.py 和 BPE_Tokenizer.py 中
from BPE_Tokenizer import BPE_Tokenizer 
from ManualEmbedding import ManualEmbedding, PositionEncoding 

# --- 全局配置 ---
D_MODEL = 64      # 向量维度
MAX_LEN = 1000    # 最大长度
VOCAB_SIZE = 1000 # 词表大小

def test_pipeline():
    print("🚀 开始全流程测试...")
    
    # 1. 初始化所有组件
    tokenizer = BPE_Tokenizer(vocab_size=VOCAB_SIZE)
    # 这一步如果是真实场景，应该先 load 训练好的 tokenizer
    # tokenizer.load("...") 
    
    emb_layer = ManualEmbedding(num_embeddings=VOCAB_SIZE, embedding_dim=D_MODEL)
    pos_layer = PositionEncoding(d_model=D_MODEL, max_len=MAX_LEN)

    # 2. 造数据
    text = "Hello world"
    print(f"\n[Step 1] 原始文本: \"{text}\"")

    # 3. Tokenizer (Str -> List)
    token_ids = tokenizer.encode(text)
    print(f"[Step 2] Token IDs: {token_ids}")

    # 4. 转换 Tensor (List -> Tensor[B, T])
    # 模拟 Batch_Size = 1
    input_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0)
    print(f"[Step 3] Input Tensor Shape: {input_tensor.shape}")

    # 5. Embedding (Tensor[B, T] -> Tensor[B, T, C])
    x_emb = emb_layer(input_tensor)
    print(f"[Step 4] Embedding Output Shape: {x_emb.shape}")
    
    # 记录一下加位置编码前的值（为了对比）
    val_before = x_emb[0, 0, :5].detach().clone()

    # 6. Position Encoding (Add PE)
    x_final = pos_layer(x_emb)
    print(f"[Step 5] Final Output Shape: {x_final.shape}")

    # --- 关键验证 ---
    val_after = x_final[0, 0, :5].detach()
    
    print("\n🔍 数值验证 (前5位):")
    print(f"  Embedding 原值: {val_before.numpy()}")
    print(f"  加上位置编码后: {val_after.numpy()}")
    
    # 检查数值是否真的变了
    if not torch.equal(val_before, val_after):
        print("\n✅ 测试通过！位置编码已成功叠加到词向量上。")
        print("   数据形状变换: [B, T] -> [B, T, D] -> [B, T, D] (形状不变，数值改变)")
    else:
        print("\n❌ 测试失败！数值没有变化，请检查 PositionEncoding 的 forward 函数是否写了 return x + self.pe")

if __name__ == '__main__':
    test_pipeline()